In [1]:
import os
import glob
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import plotly.graph_objects as go
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_groq import ChatGroq
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader

/home/elijah/elijah/ai-bc/projects/my_projects/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
knowledge_base_folder = "knowledge-base/"
vectordb_folder = "vector_db"
top_k = 2

In [3]:
model = ChatGroq(api_key=os.getenv("GROQ_API_KEY"), model_name='llama-3.3-70b-versatile')
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
sys_prompt_format = """
You are a knowledgeable, friendly teacher who teaches users artificial intelligence.
You are chatting with a user about artificial intelligence(AI).
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2985.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
def populate_vectordb():
    docs = []
    loader = DirectoryLoader(knowledge_base_folder, glob="**/*.pdf", loader_cls=PyPDFLoader)
    folder_docs = loader.load()
    for doc in folder_docs:
        docs.append(doc)
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)

    if os.path.exists(vectordb_folder):
        Chroma(persist_directory=vectordb_folder, embedding_function=vectordb_folder).delete_collection()

    db = Chroma.from_documents(documents=chunks, embedding=embedding, persist_directory=vectordb_folder)

    
    print(len(docs))
    print(len(chunks))
    return db

populate_vectordb()

370
976


In [5]:
def get_context(query):
    vector_store = Chroma(persist_directory=vectordb_folder, embedding_function=embedding).as_retriever()
    contexts = vector_store.invoke(query, k=top_k)

    contexts_str = ""
    for context in contexts:
        contexts_str += (context.page_content + "\n\n")

    return contexts_str

In [6]:
def history_to_langchain_msg_converter(history):
    messages = []
    for h in history:
        if h["role"] == "system":
            messages.append(SystemMessage(content=h["content"]))
        elif h["role"] == "user":
            messages.append(HumanMessage(content=h["content"]))
        elif h["role"] == "assistant":
            messages.append(AIMessage(content=h["content"]))
        else:
            pass
    return messages


In [7]:
def answer_question(query, history):
    context = get_context(query)
    messages = [SystemMessage(content=sys_prompt_format.format(context=context))] + history_to_langchain_msg_converter(history) + [HumanMessage(content=query)]
    res = model.invoke(messages)

    return res.content, context
    

In [8]:
ans, context = answer_question("what is AI", [])
print("="*70)
print(ans)
print("="*70)
print(context)
print("="*70)

Artificial Intelligence (AI) is the broader concept of machines being able to perform tasks that typically require human intelligence. This includes tasks such as understanding language, recognizing patterns, and making decisions.

In other words, AI refers to the development of computer systems that can perform tasks that would normally require human intelligence, such as:

* Learning from data
* Recognizing patterns
* Making decisions
* Understanding language
* Solving problems

AI is a broad field that encompasses several subfields, including Machine Learning (ML) and Deep Learning (DL). Machine Learning is a subfield of AI that focuses on developing algorithms that can learn from data and make predictions or decisions without being explicitly programmed. Deep Learning is a subfield of Machine Learning that uses neural networks with many layers to analyze data.

The goal of AI is to create machines that can think and act like humans, but we are still far from achieving true human-li

In [23]:
with gr.Blocks() as ui:
    gr.Markdown("# Machine Learning expert")
    with gr.Row():
        context = gr.Markdown("CONTEXT RETRIEVED")

        gr.ChatInterface(
        fn=answer_question,
        additional_outputs=[context]
        #additional_inputs=[model]
        )
    
    

In [24]:
ui.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
